# A2 (Colab) — 生成/收集 50 条源视频并打包回本地

这个 notebook 适用于 **本机 GPU 只有 4GB 显存** 的情况：
- 在 **Google Colab** 上生成/收集多路视频（建议 3 个生成器来源）
- 统一落盘到 `data/source/<generator>/...`
- 自动生成 `data/source_manifest.jsonl`
- 打包成 zip 下载回本地仓库根目录解压，即可用本项目做评估/标注

说明：本模板对模型/库版本做了 `try/except` 兼容；如果某个生成器在免费 T4 跑不起来，可以先跳过，后续用其他来源的视频补齐第三路。

In [ ]:
import os
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print('VRAM(GB):', round(props.total_memory / 1024**3, 2))

In [ ]:
# Colab 安装依赖（按需增删）。
# 如果你用的是 Colab 免费 T4，建议先跑一小批（例如每个生成器 2-3 条）验证可用性。
!pip -q install --upgrade pip
!pip -q install 'diffusers>=0.30.0' transformers accelerate safetensors
!pip -q install imageio imageio-ffmpeg
!pip -q install pillow

In [ ]:
from __future__ import annotations

import json
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any


OUT_ROOT = Path('videogendoctor_a2_bundle')
DATA_ROOT = OUT_ROOT / 'data'
SOURCE_ROOT = DATA_ROOT / 'source'

for sub in ['cogvideox', 'svd', 'opensora']:
    (SOURCE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print('Bundle root:', OUT_ROOT.resolve())


def write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def rel_posix(path: Path) -> str:
    # 所有路径都相对 OUT_ROOT，且用 / 作为分隔符，方便跨平台解压后直接用
    return path.relative_to(OUT_ROOT).as_posix()

## 1) 统一视频规格（强烈建议）

免费 Colab 的算力有限。为了 50 条能在一天内跑完，建议先统一规格：
- 时长：4–8 秒（建议先用 6 秒）
- 帧率：8 fps（评估够用，生成更省）
- 分辨率：>= 480p 更好；跑不动就先 512×320 或 576×320

下面会用这些默认值；你可以按需要改。

In [ ]:
DURATION_S = 6
FPS = 8
NUM_FRAMES = DURATION_S * FPS + 1

# 数量分配（建议先小批量试跑，再放大到 20/15/15）
N_COGVIDEOX = 20
N_SVD = 15
N_OPENSORA = 15

print('num_frames:', NUM_FRAMES)

## 2) 生成 prompts（50 条）

为了后续 failure taxonomy 覆盖更好，prompt 要刻意做多样性：
- 人物（单人/多人）、室内/室外、运动/静止、近景/远景
- 强约束动作（走、跑、坐下、拿起物品、交互）
- 轻量风格词（photorealistic / cinematic）

你可以直接改 `PROMPTS`（最推荐），也可以用下面的模板自动生成。

In [ ]:
random.seed(20260403)

SCENES = [
    'a small cafe',
    'a modern kitchen',
    'a busy city street',
    'a quiet office',
    'a living room',
    'a park on a sunny day',
    'a train station',
    'a supermarket aisle',
    'a classroom',
    'a gym',
]
ACTIONS = [
    'walks into the scene and sits down',
    'runs across the scene and stops',
    'picks up a cup and takes a sip',
    'opens a door and waves to the camera',
    'points at a sign and then looks back',
    'stands up, turns around, and leaves',
]
SUBJECTS = [
    'a woman in a red dress',
    'a man in a suit',
    'two friends',
    'a chef',
    'a teacher',
    'a child with a backpack',
]
STYLE = [
    'photorealistic, handheld camera, natural lighting',
    'cinematic, shallow depth of field, high detail',
    'documentary style, stable camera, realistic colors',
]

PROMPTS = []
while len(PROMPTS) < 50:
    s = random.choice(SUBJECTS)
    scene = random.choice(SCENES)
    act = random.choice(ACTIONS)
    st = random.choice(STYLE)
    p = f"{s} in {scene} {act}. {st}."
    if p not in PROMPTS:
        PROMPTS.append(p)

print('Prompts:', len(PROMPTS))
print(PROMPTS[0])

## 3) 生成器 1：CogVideoX（可选）

说明：不同环境下 `diffusers` 是否带 `CogVideoXPipeline` 不一定。
- 如果这里 import 失败：先跳过 CogVideoX，后面用其他来源补齐数量
- 如果能跑：建议先把 `N_COGVIDEOX` 改成 2–3 验证，再扩到 20

In [ ]:
def generate_cogvideox(n: int) -> list[dict[str, Any]]:
    try:
        import torch
        from diffusers.utils import export_to_video
        from diffusers import CogVideoXPipeline
    except Exception as e:
        print('CogVideoX not available in this environment:', repr(e))
        return []

    model_id = 'THUDM/CogVideoX-2b'
    out_dir = SOURCE_ROOT / 'cogvideox'

    pipe = CogVideoXPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
    # 省显存：CPU offload（需要 accelerate）
    pipe.enable_model_cpu_offload()
    try:
        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()
    except Exception:
        pass

    records: list[dict[str, Any]] = []
    for i in range(n):
        vid_id = f'real_cog_{i+1:03d}'
        prompt = PROMPTS[i]
        mp4_path = out_dir / f'{vid_id}.mp4'

        result = pipe(
            prompt=prompt,
            num_frames=NUM_FRAMES,
            guidance_scale=6.0,
        )
        frames = result.frames[0]
        export_to_video(frames, str(mp4_path), fps=FPS)

        records.append({
            'id': vid_id,
            'video_path': rel_posix(mp4_path),
            'shotir_path': None,
            'generator': 'cogvideox-2b',
            'prompt': prompt,
            'meta': {'duration_s': DURATION_S, 'fps': FPS},
        })
        print('done', vid_id)

    return records


cog_records = generate_cogvideox(N_COGVIDEOX)
print('CogVideoX videos:', len(cog_records))

## 4) 生成器 2：SVD（I2V，需要输入图）

SVD 是图生视频：你需要准备 `N_SVD` 张输入图。最省事的方法：
- 用 Colab 上传一批 jpg/png
- 或者上传一个 zip

下面先给一个“上传图片到 Colab”的 cell。

In [ ]:
# 上传 N_SVD 张图片（jpg/png）。
# 也可以先只传 2-3 张验证 pipeline 是否能跑。
from google.colab import files

uploaded = files.upload()
print('uploaded:', len(uploaded))

In [ ]:
from PIL import Image

uploaded_paths = [Path(name) for name in uploaded.keys()]
uploaded_paths = [p for p in uploaded_paths if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
print('images:', len(uploaded_paths))

# 取前 N_SVD 张
svd_images = uploaded_paths[:N_SVD]
svd_images

In [ ]:
def generate_svd(image_paths: list[Path]) -> list[dict[str, Any]]:
    try:
        import torch
        from diffusers.utils import export_to_video
        from diffusers import StableVideoDiffusionPipeline
    except Exception as e:
        print('SVD pipeline not available in this environment:', repr(e))
        return []

    # 你可以替换为你想用的 SVD checkpoint
    model_id = 'stabilityai/stable-video-diffusion-img2vid-xt'
    out_dir = SOURCE_ROOT / 'svd'

    pipe = StableVideoDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    try:
        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()
    except Exception:
        pass

    records: list[dict[str, Any]] = []
    for i, img_path in enumerate(image_paths):
        vid_id = f'real_svd_{i+1:03d}'
        mp4_path = out_dir / f'{vid_id}.mp4'

        image = Image.open(img_path).convert('RGB')
        result = pipe(
            image=image,
            num_frames=NUM_FRAMES,
        )
        frames = result.frames[0]
        export_to_video(frames, str(mp4_path), fps=FPS)

        records.append({
            'id': vid_id,
            'video_path': rel_posix(mp4_path),
            'shotir_path': None,
            'generator': 'svd-img2vid-xt',
            'prompt': None,
            'meta': {'duration_s': DURATION_S, 'fps': FPS, 'input_image': str(img_path)},
        })
        print('done', vid_id)

    return records


svd_records = generate_svd(svd_images)
print('SVD videos:', len(svd_records))

## 5) 生成器 3：Open-Sora（或任何第三路来源）

Open-Sora 生态比较分散，安装/运行脚本可能需要单独按官方仓库来。为了不把本 notebook 做死：
- 你可以在别的 Colab/环境里跑 Open-Sora
- 最终只要拿到 mp4，把它们放进本 notebook 的 `data/source/opensora/`

下面提供一个“上传一批 mp4”到 opensora 目录的 cell。

In [ ]:
# 上传第三路生成器的 mp4（例如 Open-Sora 输出）。
# 如果你还没准备好，可以先跳过，后面再补齐。
from google.colab import files

uploaded_mp4 = files.upload()
opensora_dir = SOURCE_ROOT / 'opensora'
count = 0
for name in uploaded_mp4.keys():
    p = Path(name)
    if p.suffix.lower() != '.mp4':
        continue
    target = opensora_dir / p.name
    shutil.move(str(p), str(target))
    count += 1
print('opensora mp4 moved:', count)

In [ ]:
def collect_opensora(n: int) -> list[dict[str, Any]]:
    out_dir = SOURCE_ROOT / 'opensora'
    mp4s = sorted(out_dir.glob('*.mp4'))[:n]
    records: list[dict[str, Any]] = []
    for i, mp4_path in enumerate(mp4s):
        vid_id = f'real_os_{i+1:03d}'
        # 统一命名
        target = out_dir / f'{vid_id}.mp4'
        if mp4_path.name != target.name:
            mp4_path.rename(target)
        else:
            target = mp4_path

        records.append({
            'id': vid_id,
            'video_path': rel_posix(target),
            'shotir_path': None,
            'generator': 'opensora',
            'prompt': None,
            'meta': {'duration_s': DURATION_S, 'fps': FPS},
        })
    return records


os_records = collect_opensora(N_OPENSORA)
print('OpenSora/3rd-source videos:', len(os_records))

## 6) 写 manifest + 打包下载

产物结构（解压到本地仓库根目录即可直接用）：
- `data/source/...`（mp4 文件）
- `data/source_manifest.jsonl`（清单）

注意：ShotIR 会在 A3 再补；现在 `shotir_path` 先为 null。

In [ ]:
all_records = []
all_records.extend(cog_records if 'cog_records' in globals() else [])
all_records.extend(svd_records if 'svd_records' in globals() else [])
all_records.extend(os_records if 'os_records' in globals() else [])

manifest_path = DATA_ROOT / 'source_manifest.jsonl'
write_jsonl(manifest_path, all_records)

print('total records:', len(all_records))
print('manifest:', manifest_path)

In [ ]:
# 打包成 zip 方便下载回本地
zip_base = 'videogendoctor_a2_bundle'
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=str(OUT_ROOT))
print('zip:', zip_path)

from google.colab import files
files.download(zip_path)

## 7) 本地下一步（解压后）

1) 把下载的 zip 解压到本地仓库根目录（会得到 `data/source/...` 和 `data/source_manifest.jsonl`）
2) 运行批量评估：

```bash
python infra/scripts/batch_score_manifest.py --manifest data/source_manifest.jsonl --out-root out/a2_scores
```

（可选）如果你是把 mp4 直接放到了本地 `data/source/`，也可以先用：

```bash
python infra/scripts/build_source_manifest.py --out data/source_manifest.jsonl
```

来自动重建 manifest。